# 협로 진입 규칙 실측

도크 앞 좁은 통로를 규칙 기반으로 지나기 위한 값을 **한 존씩 재서 기록**한다.

## 왜 이 값이 필요한가

| | 값 |
|---|---|
| 냉동 통로 폭 (`trihouse_map_01` 기준) | 0.20 m |
| 로봇 필요 폭 (지역 costmap, padding 포함) | 0.14 m |
| 남는 여유 | 편측 **0.03 m** |
| AMCL 위치추정 오차 (실측 stddev) | **0.08 ~ 0.11 m** |

**위치추정 오차가 통과 여유의 세 배다.** Nav2 는 절대 좌표로 계획하므로 지날 수 없다.
규칙 주행은 진입점에서의 **상대 이동**이라 AMCL 오차가 누적되지 않는다.

## 재는 것 — 존마다 넷

1. **진입점** `x, y, yaw` — 규칙 주행을 시작하는 자리
2. **존 크기** `length, width` — 그 자리를 인식할 직사각형
3. **회전 각도** — 바구니(뒤쪽 0.16 m)가 선반을 향하는 yaw
4. **후진 거리** — 도크까지 몇 m

절차 전체는 [p0-narrow-zone-measurement.md](../docs/runbooks/p0-narrow-zone-measurement.md) 에 있다.
이 노트북은 그 절차를 셀 단위로 실행하고 결과를 남기는 용도다.

> **먼저 확인할 것** — `config/narrow_zones.<지도>.yaml` 에 그 지도의 실측값이
> 이미 들어 있다. 처음부터 다시 재지 말고 **그대로 돌려 보고 어긋나는 것만** 다시 재라.

## 0. 준비

시뮬(또는 실물)이 떠 있고 AMCL 이 수렴한 상태여야 한다.

```bash
cd /home/newuser/Trihouse
scripts/p0_reset.sh && scripts/p0_up.sh
```

이 노트북은 **ROS 환경이 잡힌 상태에서** 실행해야 한다. 터미널에서:

```bash
source /opt/ros/jazzy/setup.bash && source install/setup.bash && source pinky_pro/install/setup.bash
export ROS_DOMAIN_ID=0
jupyter lab
```

In [ ]:
import math
import time
from datetime import datetime
from pathlib import Path

import rclpy
import yaml
from rclpy.node import Node
from geometry_msgs.msg import PoseWithCovarianceStamped

ROBOT = "pinky_01"
MAP_NAME = "new_map_2"                # 지금 돌고 있는 지도. 다르면 값이 안 맞는다.
REPOSITORY = Path.home() / "Trihouse"

if not rclpy.ok():
    rclpy.init()
node = Node("narrow_zone_measure")
print("ROS 노드 준비됨")

## 1. 자세를 읽는 도구

`stddev` 를 함께 본다. **0.12 m 를 넘으면 그 측정은 버린다** — 통과 여유가 0.03 m 인데
위치를 0.12 m 오차로 알면 그 값으로 만든 시퀀스는 못 쓴다. 로봇을 앞뒤로 조금 움직여
AMCL 을 다시 수렴시킨 뒤 다시 잰다.

In [ ]:
def read_pose(timeout_s: float = 5.0):
    """현재 map 프레임 자세와 위치추정 오차를 읽는다."""
    got = []
    subscription = node.create_subscription(
        PoseWithCovarianceStamped, f"/{ROBOT}/amcl_pose", got.append, 10
    )
    deadline = time.monotonic() + timeout_s
    while rclpy.ok() and time.monotonic() < deadline and not got:
        rclpy.spin_once(node, timeout_sec=0.2)
    node.destroy_subscription(subscription)
    if not got:
        raise RuntimeError("amcl_pose 가 오지 않는다 — AMCL 이 수렴하지 않았다")
    message = got[-1]
    position, orientation = message.pose.pose.position, message.pose.pose.orientation
    yaw = math.atan2(
        2 * (orientation.w * orientation.z + orientation.x * orientation.y),
        1 - 2 * (orientation.y * orientation.y + orientation.z * orientation.z),
    )
    covariance = message.pose.covariance
    reading = {
        "x": round(position.x, 6), "y": round(position.y, 6), "yaw": round(yaw, 6),
        "stddev_x": round(covariance[0] ** 0.5, 4),
        "stddev_y": round(covariance[7] ** 0.5, 4),
        "stddev_yaw": round(covariance[35] ** 0.5, 4),
    }
    print(f"x={reading['x']:.4f}  y={reading['y']:.4f}  "
          f"yaw={reading['yaw']:.4f} rad ({math.degrees(reading['yaw']):.1f} deg)")
    print(f"stddev  x={reading['stddev_x']:.3f} m  y={reading['stddev_y']:.3f} m  "
          f"yaw={reading['stddev_yaw']:.3f} rad")
    if max(reading["stddev_x"], reading["stddev_y"]) > 0.12:
        print("  ** 오차가 크다. 앞뒤로 조금 움직여 AMCL 을 다시 수렴시키고 다시 재라 **")
    return reading


read_pose()

## 2. 어느 존을 재는가

한 번에 하나씩 한다. 이름은 `destination_code` 와 같아야 한다 — 그 값으로 로봇이 존을
찾는다.

In [ ]:
ZONE = "frozen_storage_loading_dock_01"   # 상온: ambient_… / 냉장: chilled_…

record = {"zone": ZONE, "map_name": MAP_NAME, "date": datetime.now().date().isoformat()}
print("재는 존:", ZONE)

## 3. 진입점

통로 입구 **바로 앞**, 로봇이 회전할 수 있는 넓이가 남는 마지막 지점까지 수동으로
이동한 뒤 실행한다.

```bash
ros2 run teleop_twist_keyboard teleop_twist_keyboard --ros-args -r /cmd_vel:=/pinky_01/cmd_vel
```

> `teleop` 화면에서 `x` 를 여러 번 눌러 선속도를 **0.06 m/s** 아래로 내린다.

> **회전 여유 확인** — 로봇이 제자리에서 돌면 지름 **0.40 m** 원을 쓸고 지나간다
> (외접반경 0.171 m + padding 0.03). 진입점은 그만한 여유가 있어야 한다. 통로 안에서는
> 못 돈다.

In [ ]:
record["entry"] = read_pose()
record["entry"]

## 4. 바구니가 선반을 향하도록 회전

**제자리 회전만** 시킨다. 바구니(뒤쪽 긴 부분)가 선반 쪽을 향하면 멈추고 실행한다.

로봇 footprint 는 앞 0.04 m / 뒤 0.16 m 다. **긴 쪽이 바구니**이고, 그쪽이 로봇팔을
향해야 팔이 물건을 넣을 수 있다.

In [ ]:
record["rotate"] = read_pose()
print(f"\n회전 각도로 쓸 값: {record['rotate']['yaw']:.6f} rad")

## 5. 후진해 들어간다

로봇팔이 물건을 넣을 수 있는 자리까지 **후진만** 한다. 도중에 방향을 바꾸지 않는다.

In [ ]:
record["docked"] = read_pose()

entry, docked = record["entry"], record["docked"]
distance = math.hypot(docked["x"] - entry["x"], docked["y"] - entry["y"])
record["reverse_distance"] = round(distance, 4)
print(f"\n후진 거리: {distance:.4f} m")

## 6. 도크 실측 좌표와 대조

`config` 가 아니라 **승인된 JSONL** 의 도크 좌표와 견준다. 규칙 주행이 끝난 자리가
도크와 얼마나 다른지가 곧 "바구니가 팔에 닿는가" 다.

`map_yaw` 는 실측 기록에 **"pinky 방향(바구니 방향 판단용, 중요)"** 으로 적혀 있다.

In [ ]:
FEATURES = REPOSITORY / "control_ui/rmf_control_ui/data/import/trihouse_test_01_physical_features.jsonl"

import json as _json
dock = None
for line in FEATURES.read_text(encoding="utf-8").splitlines():
    line = line.strip()
    if not line:
        continue
    entry_record = _json.loads(line)
    if entry_record.get("rmf_waypoint_name") == ZONE:
        dock = entry_record["map_pose"]
        break

if dock is None:
    print(f"{ZONE} 의 실측 좌표를 JSONL 에서 찾지 못했다")
else:
    d = math.hypot(record["docked"]["x"] - dock["x"], record["docked"]["y"] - dock["y"])
    yaw_error = abs(math.atan2(
        math.sin(record["docked"]["yaw"] - dock["yaw"]),
        math.cos(record["docked"]["yaw"] - dock["yaw"]),
    ))
    print(f"도크 실측  x={dock['x']:.3f} y={dock['y']:.3f} yaw={dock['yaw']:.3f}")
    print(f"지금 자리  x={record['docked']['x']:.3f} y={record['docked']['y']:.3f} "
          f"yaw={record['docked']['yaw']:.3f}")
    print(f"\n거리 {d:.3f} m   yaw 차 {yaw_error:.3f} rad ({math.degrees(yaw_error):.1f} deg)")
    print(f"판정: {'통과' if d <= 0.15 and yaw_error <= 0.35 else '**허용오차 밖**'} "
          f"(기준 0.15 m / 0.35 rad)")
    record["dock_reference"] = dock

## 7. 존 직사각형

`length` 는 진행축 방향으로 판정할 폭, `width` 는 **실측한 통로 폭**이다.
통로 폭은 지도에서 확인할 수 있다.

```bash
scripts/p0_show_map.py
```

출력 끝의 도달 가능성 표에 존별 최선 통로 폭이 나온다.

In [ ]:
record["zone_shape"] = {"length": 0.10, "width": 0.20}   # 실측값으로 바꾼다
record["zone_shape"]

## 8. 나오는 시퀀스

입고의 역순이지만 각도는 다르다. 후진했던 만큼 전진 → 회전 → 존을 벗어날 때까지 전진.
회전을 마친 자리에서 실행한다.

In [ ]:
record["exit_rotate"] = read_pose()
print(f"\n탈출 회전 각도: {record['exit_rotate']['yaw']:.6f} rad")

## 9. 존 표 조각을 만든다

출력된 것을 `config/narrow_zones.<지도>.yaml` 의 해당 존에 붙여 넣는다.
**측정 이력(`measured`)을 함께 남긴다** — 나중에 값이 의심스러울 때 되돌아갈 근거다.

In [ ]:
enter = []
if abs(record.get("forward_before_rotate", 0.0)) > 0.001:
    enter.append(["straight", round(record["forward_before_rotate"], 4)])
enter.append(["rotate", record["rotate"]["yaw"]])
enter.append(["straight", -record["reverse_distance"]])

fragment = {
    ZONE: {
        "entry": {k: record["entry"][k] for k in ("x", "y", "yaw")},
        "zone": record["zone_shape"],
        "enter": enter,
        "exit": [
            ["straight", record["reverse_distance"]],
            ["rotate", record["exit_rotate"]["yaw"]],
            ["exit_zone", None],
        ],
        "measured": {
            "date": record["date"],
            "source": "notebook",
            "stddev_x": record["entry"]["stddev_x"],
            "stddev_y": record["entry"]["stddev_y"],
            "stddev_yaw": record["entry"]["stddev_yaw"],
            "note": "무엇을 보고 이 값으로 정했는지 여기에 적는다",
        },
    }
}
print(yaml.safe_dump(fragment, allow_unicode=True, sort_keys=False))

## 10. 원자료를 남긴다

셀 출력은 노트북을 다시 실행하면 사라진다. 측정 원자료는 파일로 남긴다.

In [ ]:
out = REPOSITORY / "docs" / "calibration" / f"narrow_zone_{ZONE}_{record['date']}.json"
out.parent.mkdir(parents=True, exist_ok=True)
out.write_text(json.dumps(record, ensure_ascii=False, indent=2), encoding="utf-8")
print("남김:", out)

## 11. 검증

존 표를 갱신했으면 시뮬에서 확인한다.

```bash
scripts/p0_reset.sh && scripts/p0_up.sh
```

로그에서 볼 것:

```bash
grep -a "협로" /tmp/sim.log
```

| 나와야 하는 줄 | 뜻 |
|---|---|
| `협로 존 N개 적재` | 표를 읽었다. 지도 이름이 맞다 |
| `협로 존 진입점으로 먼저 간다` | Nav2 가 진입점까지 간다 |
| `협로 enter 1/3 — straight 0.1` | 시퀀스가 스텝별로 돈다 |
| `협로 진입 후 도크와 거리 … yaw 차 …` | **바구니가 팔에 닿는 자리인가** |

거리가 0.15 m, yaw 차가 0.35 rad 를 넘으면 단계가 실패한다. 그 값을 보고 어느 스텝을
다시 재야 하는지 판단한다.

---

## 이것이 임시라는 것

규칙 주행은 좌표가 지도에 묶여 있고 되먹임이 없다. 최종형은
[마커 기반 도킹](../docs/architecture/marker-docking-design.md)이다 — 마커 상대
좌표로 정렬하므로 지도가 바뀌어도 동작하고, 되먹임이 있어 드리프트에도 강하다.
**도킹이 붙으면 이 측정값은 걷어낸다.**